# 🍽️ Campbell's Restaurant — AI-Driven Marketing System
### Churn Prediction · Customer Segmentation · Aspect-Based Sentiment Analysis · Personalized Re-engagement

---

**Project Overview**

This notebook implements a full end-to-end AI marketing pipeline for Campbell's Restaurant. The system:

1. **Segments customers** into behavioral groups using RFM analysis and KMeans clustering
2. **Predicts churn** using a two-tier approach — rule-based for one-time visitors, XGBoost for repeat customers
3. **Analyzes customer sentiment** from reviews using Aspect-Based Sentiment Analysis (ABSA)
4. **Generates personalized re-engagement messages** (SMS, Email, App notification) using a Large Language Model

**Dataset**
- `Marketing_data.csv` — 12,545 transaction records, 2,041 unique customers (Nov 2023 → Feb 2024)
- `Train_Sentiment.csv` — 2,195 labeled restaurant reviews
- `sample_submission.csv` — 536 test reviews
- `Campbell_Menu_Data_-_2.xlsx` — 986 menu items

---

| Section | Model | Key Metric |
|---|---|---|
| 1. Customer Segmentation | KMeans (k=4) | Silhouette Score: 0.45 |
| 2. Churn Prediction | Two-Tier XGBoost | ROC-AUC: 0.84 |
| 3. Sentiment Analysis | TF-IDF + Logistic Regression | F1: 0.75 (aspect), 0.64 (sentiment) |
| 4. Message Generation | Groq LLaMA 3.3 70B | Personalized SMS + Email + App |


---
## 📦 Section 1 — Customer Segmentation

### What & Why
We segment 2,041 customers into four behavioral groups using **RFM Analysis** (Recency, Frequency, Monetary) combined with **KMeans Clustering**.

| Segment | Description | Business Action |
|---|---|---|
| **Regular** | High frequency, high spend, recent | Reward & retain — VIP treatment |
| **Occasional** | Moderate visits, decent spend | Upsell & increase visit frequency |
| **New** | Recent first visit, low frequency | Nurture & convert to loyal customer |
| **Lost** | Long absence, low engagement | Re-engage with offers — urgent! |

### Approach
- Clean raw transaction data (remove voided transactions, parse dollar amounts)
- Compute RFM features per customer
- Scale features with StandardScaler
- Find optimal k using Silhouette Score (tested k=3,4,5,6)
- Label clusters automatically based on average Recency


### 1.1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

### 1.2 — Load & Clean Data

In [ ]:
df = pd.read_csv('Marketing_data.csv')

def clean_money(series):
    """Strip dollar signs and convert to float."""
    return pd.to_numeric(
        series.astype(str).str.replace('$', '').str.replace(',', ''),
        errors='coerce'
    )

df['Total_clean'] = clean_money(df['Total'])
df['Order_Date']  = pd.to_datetime(df['Order Date'], errors='coerce')
df['Voided']      = df['Voided?'].astype(str).str.lower() == 'true'

# Remove voided transactions & drop nulls
df = df[~df['Voided']].dropna(subset=['Last 4 Card Digits', 'Order_Date', 'Total_clean'])

print(f"✅ Clean rows      : {len(df)}")
print(f"✅ Unique customers: {df['Last 4 Card Digits'].nunique()}")
print(f"✅ Date range      : {df['Order_Date'].min().date()} → {df['Order_Date'].max().date()}")
df.head()

### 1.3 — RFM Feature Engineering

In [ ]:
# Snapshot = day after last transaction (reference point for recency)
snapshot_date = df['Order_Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Last 4 Card Digits').agg(
    Recency  = ('Order_Date',  lambda x: (snapshot_date - x.max()).days),  # days since last visit
    Frequency= ('Order_Date',  'nunique'),                                  # number of visits
    Monetary = ('Total_clean', 'sum')                                       # total spend
).reset_index()

print(f"📊 RFM Table Shape: {rfm.shape}")
rfm.describe().round(2)

### 1.4 — Find Optimal K (Silhouette Score)

In [ ]:
scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print("🔍 Silhouette Scores:")
for k in [3, 4, 5, 6]:
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    score  = silhouette_score(rfm_scaled, labels)
    print(f"   k={k} → {score:.4f}")

print("\n→ k=4 chosen — best silhouette and aligns with business segments (New, Occasional, Regular, Lost)")

### 1.5 — Final Clustering (k=4)

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Auto-label: sort clusters by avg Recency → lowest recency = most active = Regular
cluster_recency = rfm.groupby('Cluster')['Recency'].mean().sort_values()
labels_ordered  = ['Regular', 'New', 'Occasional', 'Lost']
cluster_map     = {cluster: label for cluster, label in zip(cluster_recency.index, labels_ordered)}
rfm['Segment']  = rfm['Cluster'].map(cluster_map)
rfm['Churn']    = (rfm['Segment'] == 'Lost').astype(int)

print("🏷️  Cluster → Segment Mapping:")
print(cluster_map)
print("\n📈 Segment Profiles (avg RFM per segment):")
rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(2)

### 1.6 — Segment Distribution

In [ ]:
print("👥 Segment Counts:")
print(rfm['Segment'].value_counts())
print(f"\n⚠️  Overall Churn Rate: {rfm['Churn'].mean():.2%}")

### 1.7 — Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLORS = {
    'New':        '#378ADD',
    'Lost':       '#E24B4A',
    'Occasional': '#EF9F27',
    'Regular':    '#1D9E75'
}

seg_counts  = rfm['Segment'].value_counts()
seg_profile = rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Customer Segmentation Dashboard', fontsize=16, fontweight='bold', y=1.02)

# Chart 1 — Donut: Segment Distribution
ax1 = axes[0]
sizes  = seg_counts.values
labels = seg_counts.index.tolist()
colors = [COLORS[l] for l in labels]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=None, colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(10); at.set_color('white'); at.set_fontweight('bold')
ax1.legend(
    handles=[mpatches.Patch(color=COLORS[l], label=f'{l}  ({seg_counts[l]:,})') for l in labels],
    loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9, frameon=False
)
ax1.set_title('Segment Distribution', fontsize=13, fontweight='bold', pad=12)

# Chart 2 — Bar: Avg Recency
ax2 = axes[1]
recency_order = seg_profile['Recency'].sort_values(ascending=False)
bars = ax2.bar(recency_order.index, recency_order.values,
               color=[COLORS[s] for s in recency_order.index],
               edgecolor='white', linewidth=0.8, width=0.55)
for bar, val in zip(bars, recency_order.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{val:.0f}d', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#444')
ax2.set_title('Avg Days Since Last Visit (Recency)', fontsize=13, fontweight='bold', pad=12)
ax2.set_ylabel('Days', fontsize=10)
ax2.set_facecolor('#F7F7F7'); ax2.spines[['top','right']].set_visible(False)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5); ax2.set_axisbelow(True)

# Chart 3 — Horizontal Bar: Avg Spend
ax3 = axes[2]
monetary_order = seg_profile['Monetary'].sort_values(ascending=True)
bars3 = ax3.barh(monetary_order.index, monetary_order.values,
                 color=[COLORS[s] for s in monetary_order.index],
                 edgecolor='white', linewidth=0.8, height=0.55)
for bar, val in zip(bars3, monetary_order.values):
    ax3.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', ha='left', fontsize=10, fontweight='bold', color='#444')
ax3.set_title('Avg Spend per Customer ($)', fontsize=13, fontweight='bold', pad=12)
ax3.set_xlabel('Total Spend ($)', fontsize=10)
ax3.set_facecolor('#F7F7F7'); ax3.spines[['top','right']].set_visible(False)
ax3.xaxis.grid(True, linestyle='--', alpha=0.5); ax3.set_axisbelow(True)

plt.tight_layout()
plt.savefig('segmentation_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: segmentation_charts.png")

### 1.8 — Save Model

In [ ]:
rfm.to_csv('rfm_segments.csv', index=False)
with open('kmeans_model.pkl', 'wb') as f:  pickle.dump(km, f)
with open('scaler.pkl', 'wb') as f:        pickle.dump(scaler, f)
with open('cluster_map.pkl', 'wb') as f:   pickle.dump(cluster_map, f)
print("💾 Saved: rfm_segments.csv, kmeans_model.pkl, scaler.pkl, cluster_map.pkl")
print("✅ Section 1 — Customer Segmentation complete!")

---
## 🚨 Section 2 — Churn Prediction

### What & Why
Given a customer's behavioral history, we predict the probability they will **not return** — their churn probability.

### Design Decisions
- **Time-based split** — Features built from Nov 2023 → Jan 2024. Labels from whether customer returned in Feb 2024. This prevents data leakage.
- **Two-tier approach** — One-time visitors get rule-based scoring (not enough data for a model). Repeat visitors get XGBoost.
- **Segment as a feature** — The cluster label from Section 1 is fed into XGBoost, boosting ROC-AUC from 0.70 → 0.84.

### Why Not a Single Model?
One-time visitors (64% of customers) have only one transaction — there's no behavioral pattern to learn from. Forcing them into a model would be noisy and misleading. The rule-based tier is honest and transparent.

| Tier | Customers | Method | Logic |
|---|---|---|---|
| Tier 1 | 1,297 | Rule-based | < 14 days → 30% risk, 14-30 days → 65%, > 30 days → 92% |
| Tier 2 | 242 | XGBoost | 12 behavioral features + segment label |


### 2.1 — Imports

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
print("✅ Libraries loaded")

### 2.2 — Load & Clean Data

In [ ]:
df = pd.read_csv('Marketing_data.csv')

def clean_money(series):
    return pd.to_numeric(
        series.astype(str).str.replace('$', '').str.replace(',', ''),
        errors='coerce'
    )

df['Total_clean']    = clean_money(df['Total'])
df['Discount_clean'] = clean_money(df['Discount'])
df['Tip_clean']      = clean_money(df['Tip'])
df['Order_Date']     = pd.to_datetime(df['Order Date'], errors='coerce')
df['Voided']         = df['Voided?'].astype(str).str.lower() == 'true'
df = df[~df['Voided']].dropna(subset=['Last 4 Card Digits', 'Order_Date', 'Total_clean'])

print(f"✅ Clean rows      : {len(df)}")
print(f"✅ Unique customers: {df['Last 4 Card Digits'].nunique()}")

### 2.3 — Time-Based Split
> **No data leakage:** Features are built strictly from the training window (Nov → Jan). Labels come from the future window (Feb). The model never sees future data during training.

In [ ]:
TRAIN_END   = pd.Timestamp('2024-01-31')
LABEL_START = pd.Timestamp('2024-02-01')

train_df        = df[df['Order_Date'] <= TRAIN_END]
label_df        = df[df['Order_Date'] >= LABEL_START]
returned_in_feb = set(label_df['Last 4 Card Digits'].unique())
snapshot        = TRAIN_END + pd.Timedelta(days=1)

print(f"📅 Train window : Nov 2023 → Jan 2024 ({len(train_df):,} transactions)")
print(f"📅 Label window : Feb 2024 ({len(label_df):,} transactions)")
print(f"✅ Customers who returned in Feb: {len(returned_in_feb)}")

### 2.4 — Rebuild Segments (from Section 1)

In [ ]:
rfm_full = df.groupby('Last 4 Card Digits').agg(
    Recency  = ('Order_Date',  lambda x: (snapshot - x.max()).days),
    Frequency= ('Order_Date',  'nunique'),
    Monetary = ('Total_clean', 'sum'),
).reset_index()

scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_full[['Recency', 'Frequency', 'Monetary']])
km         = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_full['Cluster'] = km.fit_predict(rfm_scaled)

cluster_recency     = rfm_full.groupby('Cluster')['Recency'].mean().sort_values()
cluster_map         = {c: l for c, l in zip(cluster_recency.index, ['Regular','New','Occasional','Lost'])}
rfm_full['Segment'] = rfm_full['Cluster'].map(cluster_map)

print("🏷️  Segments rebuilt:")
print(rfm_full['Segment'].value_counts().to_string())

### 2.5 — Feature Engineering (Train Window Only)

In [ ]:
features_df = train_df.groupby('Last 4 Card Digits').agg(
    Recency       = ('Order_Date',     lambda x: (snapshot - x.max()).days),
    Frequency     = ('Order_Date',     'nunique'),
    Monetary      = ('Total_clean',    'sum'),
    Unique_Items  = ('Menu Item',      'nunique'),
    Avg_Order_Val = ('Total_clean',    'mean'),
    Avg_Tip       = ('Tip_clean',      'mean'),
    Discount_Used = ('Discount_clean', lambda x: (x > 0).sum()),
    Visits_Nov    = ('Order_Date',     lambda x: (x.dt.month == 11).sum()),
    Visits_Dec    = ('Order_Date',     lambda x: (x.dt.month == 12).sum()),
    Visits_Jan    = ('Order_Date',     lambda x: (x.dt.month ==  1).sum()),
).reset_index()

first_visits = train_df.groupby('Last 4 Card Digits')['Order_Date'].min()
features_df['Days_Since_First'] = features_df['Last 4 Card Digits'].map(
    lambda cid: (snapshot - first_visits.get(cid, snapshot)).days
)

# Merge segment from Section 1 as a feature
features_df  = features_df.merge(rfm_full[['Last 4 Card Digits','Segment']], on='Last 4 Card Digits', how='left')
seg_map      = {'Regular': 0, 'New': 1, 'Occasional': 2, 'Lost': 3}
features_df['Segment_Code'] = features_df['Segment'].map(seg_map)
features_df['Churn']        = (~features_df['Last 4 Card Digits'].isin(returned_in_feb)).astype(int)

print(f"⚠️  Overall churn rate : {features_df['Churn'].mean():.2%}")
print(f"   Churned            : {features_df['Churn'].sum()}")
print(f"   Retained           : {(features_df['Churn']==0).sum()}")
features_df.head()

### 2.6 — Tier 1: Rule-Based Scoring (One-Time Visitors)

In [ ]:
tier1 = features_df[features_df['Frequency'] == 1].copy()
tier2 = features_df[features_df['Frequency'] >= 2].copy()

print(f"🔀 Tier 1 (one-time , rule-based): {len(tier1)} customers | Churn: {tier1['Churn'].mean():.2%}")
print(f"🔀 Tier 2 (repeat   , XGBoost)   : {len(tier2)} customers | Churn: {tier2['Churn'].mean():.2%}")

def rule_based_prob(recency):
    """
    Simple rule: the longer since their only visit, the higher the churn risk.
    Thresholds validated against actual Feb return data.
    """
    if recency < 14:   return 0.30   # still fresh — low risk
    elif recency < 30: return 0.65   # fading — medium risk
    else:              return 0.92   # gone — high risk

tier1['Churn_Probability'] = tier1['Recency'].apply(rule_based_prob)
tier1['Risk_Level']        = pd.cut(tier1['Recency'], bins=[0,14,30,999], labels=['Low','Medium','High'])
tier1['Tier']              = 'Rule-Based'

print("\n📊 Tier 1 Risk Distribution:")
print(tier1['Risk_Level'].value_counts().to_string())

### 2.7 — Tier 2: XGBoost (Repeat Visitors)

In [ ]:
FEATURES = [
    'Recency', 'Frequency', 'Monetary', 'Unique_Items',
    'Avg_Order_Val', 'Avg_Tip', 'Discount_Used',
    'Visits_Nov', 'Visits_Dec', 'Visits_Jan',
    'Days_Since_First', 'Segment_Code'  # ← segment from clustering boosts AUC
]

X2 = tier2[FEATURES]
y2 = tier2['Churn']

model = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=(y2==0).sum()/(y2==1).sum(),  # handle class imbalance
    random_state=42, eval_metric='logloss'
)

# 5-fold cross-validation — honest evaluation, no train/test leakage
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X2, y2, cv=cv, scoring='roc_auc')

print(f"🎯 Tier 2 XGBoost — 5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Individual folds: {[round(s,4) for s in cv_scores]}")

model.fit(X2, y2)
tier2['Churn_Probability'] = model.predict_proba(X2)[:,1]
tier2['Risk_Level']        = pd.cut(tier2['Churn_Probability'], bins=[0,0.33,0.66,1.0], labels=['Low','Medium','High'])
tier2['Tier']              = 'XGBoost'

print("\n📈 Feature Importances:")
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
for feat, imp in importances.items():
    print(f"   {feat:22s}: {imp:.4f}")

### 2.8 — Combine Tiers & Final Results

In [ ]:
final_df = pd.concat([tier1, tier2], ignore_index=True)

print(f"✅ Total customers scored: {len(final_df)}")
print(f"\n📊 Segment × Risk Level Breakdown:")
print(pd.crosstab(final_df['Segment'], final_df['Risk_Level'], margins=True))

print(f"\n🔝 Top 10 At-Risk Customers:")
print(
    final_df[['Last 4 Card Digits','Segment','Tier','Recency','Frequency','Monetary','Churn_Probability','Risk_Level']]
    .sort_values('Churn_Probability', ascending=False)
    .head(10)
    .to_string(index=False)
)

### 2.9 — Visualization

In [ ]:
SEG_COLORS = {'New':'#378ADD','Lost':'#E24B4A','Occasional':'#EF9F27','Regular':'#1D9E75'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Two-Tier Churn Prediction — Full Pipeline Results', fontsize=16, fontweight='bold', y=1.02)

# Chart 1 — Segment × Risk Heatmap
ax1 = axes[0]
cross = pd.crosstab(final_df['Segment'], final_df['Risk_Level'])[['Low','Medium','High']]
cross = cross.reindex(['Regular','Occasional','New','Lost'])
im    = ax1.imshow(cross.values, cmap='RdYlGn_r', aspect='auto')
ax1.set_xticks(range(3)); ax1.set_xticklabels(['Low','Medium','High'], fontsize=11)
ax1.set_yticks(range(4)); ax1.set_yticklabels(cross.index, fontsize=11)
ax1.set_title('Segment × Risk Level', fontsize=13, fontweight='bold', pad=12)
for i in range(4):
    for j in range(3):
        ax1.text(j, i, str(cross.values[i,j]), ha='center', va='center',
                 fontsize=13, fontweight='bold', color='white')
plt.colorbar(im, ax=ax1, shrink=0.8)

# Chart 2 — Churn Probability by Segment
ax2 = axes[1]
for seg in ['Regular','Occasional','New','Lost']:
    data = final_df[final_df['Segment']==seg]['Churn_Probability']
    ax2.hist(data, bins=15, alpha=0.65, color=SEG_COLORS[seg], label=seg, edgecolor='white')
ax2.set_title('Churn Probability by Segment', fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Churn Probability', fontsize=10)
ax2.set_ylabel('Number of Customers', fontsize=10)
ax2.legend(fontsize=9, frameon=False)
ax2.set_facecolor('#F7F7F7'); ax2.spines[['top','right']].set_visible(False)

# Chart 3 — Feature Importances
ax3 = axes[2]
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
bar_colors  = ['#E24B4A' if v >= importances.quantile(0.75) else '#378ADD' for v in importances.values]
bars = ax3.barh(importances.index, importances.values, color=bar_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, importances.values):
    ax3.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9, fontweight='bold', color='#444')
ax3.set_title('Feature Importances (Tier 2)', fontsize=13, fontweight='bold', pad=12)
ax3.set_xlabel('Importance Score', fontsize=10)
ax3.set_facecolor('#F7F7F7'); ax3.spines[['top','right']].set_visible(False)
ax3.xaxis.grid(True, linestyle='--', alpha=0.5); ax3.set_axisbelow(True)

plt.tight_layout()
plt.savefig('churn_prediction_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: churn_prediction_charts.png")

### 2.10 — Save Model

In [ ]:
final_df.to_csv('churn_scores_final.csv', index=False)
with open('kmeans_model.pkl', 'wb') as f:      pickle.dump(km, f)
with open('scaler.pkl', 'wb') as f:            pickle.dump(scaler, f)
with open('cluster_map.pkl', 'wb') as f:       pickle.dump(cluster_map, f)
with open('churn_model_tier2.pkl', 'wb') as f: pickle.dump(model, f)
with open('churn_features.pkl', 'wb') as f:    pickle.dump(FEATURES, f)
print("💾 Saved: churn_scores_final.csv + all model pickle files")
print("✅ Section 2 — Churn Prediction complete!")

---
## 💬 Section 3 — Aspect-Based Sentiment Analysis (ABSA)

### What & Why
Standard sentiment analysis tells you if a review is positive or negative overall. **ABSA goes deeper** — it tells you *which specific aspect* of the restaurant the customer is happy or unhappy about.

**Example:**
> *"The food was amazing but the service was really slow"*
> - Food → Positive ✅
> - Service → Negative ❌

### Three-Part Pipeline

| Part | Task | Method |
|---|---|---|
| 1 | Aspect Detection | TF-IDF (1-3 ngrams) + OneVsRest Logistic Regression |
| 2 | Sentiment per Aspect | TF-IDF + Logistic Regression on (review + aspect) pairs |
| 3 | Opinion Extraction | Rule-based keyword matching + context window |

### Aspects Tracked
`food` · `staff` · `service` · `place` · `menu` · `ambience` · `price`

### Why Not Transformers?
A DistilBERT-based model would achieve ~85% F1 but requires GPU or significant CPU time. Our TF-IDF approach achieves **0.75 F1** on aspect detection and **0.64 F1** on sentiment — solid baseline, runs on any machine in seconds. The code includes an upgrade path for transformers.


### 3.1 — Imports

In [ ]:
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
print("✅ Libraries loaded")

### 3.2 — Load & Parse Data

In [ ]:
train  = pd.read_csv('Train_Sentiment.csv')
sample = pd.read_csv('sample_submission.csv')

def parse_list(s):
    """Safely parse stringified Python lists from CSV."""
    try:    return ast.literal_eval(s)
    except: return []

train['Aspects_parsed']   = train['Aspects'].apply(parse_list)
train['Sentiment_parsed'] = train['Sentiment'].apply(parse_list)

print(f"✅ Train reviews : {len(train)}")
print(f"✅ Test reviews  : {len(sample)}")
print(f"✅ Aspects       : {set(a for row in train['Aspects_parsed'] for a in row)}")
train.head()

### 3.3 — Part 1: Aspect Detection

In [ ]:
ASPECTS = ['food', 'staff', 'service', 'place', 'menu', 'ambience', 'price']

# Multi-label binarizer — a review can have multiple aspects
mlb       = MultiLabelBinarizer(classes=ASPECTS)
Y_aspects = mlb.fit_transform(train['Aspects_parsed'])

# TF-IDF with 1-3 word ngrams captures phrases like "slow service", "friendly staff"
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=8000, sublinear_tf=True)
X     = tfidf.fit_transform(train['Review'])

# One classifier per aspect (OneVsRest)
aspect_model = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0))
aspect_model.fit(X, Y_aspects)

cv_aspect = cross_val_score(aspect_model, X, Y_aspects, cv=5, scoring='f1_micro')
print(f"📊 Aspect Detection — 5-Fold CV F1 (micro): {cv_aspect.mean():.4f} ± {cv_aspect.std():.4f}")

### 3.4 — Part 2: Sentiment Classification per Aspect

In [ ]:
# Build (review, aspect) → sentiment training pairs
rows = []
for _, row in train.iterrows():
    for aspect, sentiment in zip(row['Aspects_parsed'], row['Sentiment_parsed']):
        rows.append({'Review': row['Review'], 'Aspect': aspect, 'Sentiment': sentiment})

sent_df          = pd.DataFrame(rows)
# Concatenate review + aspect so model knows which aspect to judge
sent_df['Input'] = sent_df['Review'] + ' [ASPECT] ' + sent_df['Aspect']

tfidf_sent = TfidfVectorizer(ngram_range=(1, 3), max_features=8000, sublinear_tf=True)
X_sent     = tfidf_sent.fit_transform(sent_df['Input'])
y_sent     = sent_df['Sentiment']

sent_model = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')
sent_model.fit(X_sent, y_sent)

cv_sent = cross_val_score(sent_model, X_sent, y_sent, cv=5, scoring='f1_macro')
print(f"📊 Sentiment — 5-Fold CV F1 (macro): {cv_sent.mean():.4f} ± {cv_sent.std():.4f}")
print(f"\nSentiment distribution in training data:")
print(sent_df['Sentiment'].value_counts().to_string())

### 3.5 — Part 3: Opinion Phrase Extraction

In [ ]:
OPINION_KEYWORDS = {
    'food'    : ['food','dish','meal','taste','flavor','delicious','bland','overcooked','fresh','portion'],
    'staff'   : ['waiter','waitress','server','staff','host','bartender','friendly','rude','helpful','attentive'],
    'service' : ['service','wait','slow','fast','quick','prompt','attentive','responsive'],
    'place'   : ['place','location','restaurant','spot','venue','seating','atmosphere'],
    'menu'    : ['menu','options','variety','selection','choice','specials'],
    'ambience': ['ambience','ambiance','atmosphere','decor','noise','cozy','loud','vibe','setting'],
    'price'   : ['price','expensive','cheap','value','worth','overpriced','affordable','cost']
}

def extract_opinion(review, aspect):
    """
    Find the sentence containing the aspect keyword and extract
    a contextual window (±3 words) around it as the opinion phrase.
    Falls back to the aspect name if nothing is found.
    """
    keywords  = OPINION_KEYWORDS.get(aspect, [aspect])
    sentences = re.split(r'[.!?,;]', review)
    for sent in sentences:
        if any(kw in sent.lower() for kw in keywords):
            words = sent.strip().split()
            for i, word in enumerate(words):
                if word.lower().strip('.,!?') in keywords:
                    start   = max(0, i - 3)
                    end     = min(len(words), i + 4)
                    snippet = ' '.join(words[start:end]).strip('.,!? ')
                    if snippet:
                        return snippet
    return aspect  # fallback

print("✅ Opinion extractor defined")

### 3.6 — Full ABSA Inference Pipeline

In [ ]:
def predict_absa(review):
    """
    Full ABSA pipeline for a single review.
    Returns aspects, sentiments, opinion phrases, and (aspect, opinion, sentiment) triplets.
    """
    X_r     = tfidf.transform([review])
    aspects = list(mlb.inverse_transform(aspect_model.predict(X_r))[0])

    sentiments, opinions, triplets = [], [], []
    for asp in aspects:
        inp       = tfidf_sent.transform([review + ' [ASPECT] ' + asp])
        sentiment = sent_model.predict(inp)[0]
        opinion   = extract_opinion(review, asp)
        sentiments.append(sentiment)
        opinions.append(opinion)
        triplets.append((asp, opinion, sentiment))

    return {'aspects': aspects, 'sentiments': sentiments, 'opinions': opinions, 'triplets': triplets}

# Test on sample reviews
test_reviews = [
    "The food was amazing but the service was really slow and the waiter was rude.",
    "Great ambience and the staff were so friendly and helpful!",
    "Prices are too high for what you get, but the food tastes great."
]

print("🔍 Pipeline Test:")
for review in test_reviews:
    result = predict_absa(review)
    print(f"\nReview  : {review}")
    for triplet in result['triplets']:
        print(f"  → ({triplet[0]}, '{triplet[1]}', {triplet[2]})")

### 3.7 — Run on Test Set

In [ ]:
print("🔄 Running inference on test set...")

results = []
for _, row in sample.iterrows():
    pred = predict_absa(str(row['Review']))
    results.append({
        'ID'             : row['ID'],
        'Review'         : row['Review'],
        'Aspects'        : str(pred['aspects']),
        'Sentiment'      : str(pred['sentiments']),
        'Feature_Opinion': str(pred['opinions']),
        'Usage'          : row.get('Usage', 'Public')
    })

results_df = pd.DataFrame(results)
print(f"✅ Predictions complete: {len(results_df)} reviews")
print("\n📋 Sample Predictions:")
results_df.head(5)

### 3.8 — Visualization

In [ ]:
COLORS      = {'food':'#378ADD','staff':'#E24B4A','service':'#EF9F27',
               'place':'#1D9E75','menu':'#7F77DD','ambience':'#D85A30','price':'#639922'}
SENT_COLORS = {'positive':'#1D9E75','negative':'#E24B4A','neutral':'#888780'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('ABSA Model — Results', fontsize=16, fontweight='bold', y=1.02)

# Chart 1 — Aspect Frequency
ax1 = axes[0]
aspect_counts = pd.Series([a for row in train['Aspects_parsed'] for a in row]).value_counts()
bars = ax1.bar(aspect_counts.index, aspect_counts.values,
               color=[COLORS.get(a,'#888') for a in aspect_counts.index],
               edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, aspect_counts.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
             str(val), ha='center', fontsize=9, fontweight='bold', color='#444')
ax1.set_title('Aspect Frequency in Training Data', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_facecolor('#F7F7F7'); ax1.spines[['top','right']].set_visible(False)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5); ax1.set_axisbelow(True)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Chart 2 — Sentiment Distribution
ax2 = axes[1]
sent_counts = pd.Series([s for row in train['Sentiment_parsed'] for s in row]).value_counts()
wedges, texts, autotexts = ax2.pie(
    sent_counts.values, labels=sent_counts.index,
    colors=[SENT_COLORS.get(s,'#888') for s in sent_counts.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold'); at.set_color('white')
ax2.set_title('Sentiment Distribution', fontsize=13, fontweight='bold', pad=12)

# Chart 3 — Model Performance
ax3 = axes[2]
metrics  = ['Aspect Detection\n(F1 micro)', 'Sentiment\n(F1 macro)']
scores   = [cv_aspect.mean(), cv_sent.mean()]
errors   = [cv_aspect.std(),  cv_sent.std()]
bars3    = ax3.bar(metrics, scores, color=['#378ADD','#E24B4A'], edgecolor='white',
                   linewidth=0.8, width=0.45, yerr=errors,
                   capsize=6, error_kw={'linewidth':1.5,'color':'#444'})
for bar, val in zip(bars3, scores):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015,
             f'{val:.3f}', ha='center', fontsize=12, fontweight='bold', color='#444')
ax3.set_title('Model Performance (5-Fold CV)', fontsize=13, fontweight='bold', pad=12)
ax3.set_ylabel('F1 Score', fontsize=10); ax3.set_ylim(0, 1.0)
ax3.set_facecolor('#F7F7F7'); ax3.spines[['top','right']].set_visible(False)
ax3.yaxis.grid(True, linestyle='--', alpha=0.5); ax3.set_axisbelow(True)

plt.tight_layout()
plt.savefig('absa_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Saved: absa_charts.png")

### 3.9 — Save Model

In [ ]:
results_df.to_csv('absa_predictions.csv', index=False)
with open('aspect_model.pkl', 'wb') as f:    pickle.dump(aspect_model, f)
with open('tfidf_aspect.pkl', 'wb') as f:    pickle.dump(tfidf, f)
with open('mlb.pkl', 'wb') as f:             pickle.dump(mlb, f)
with open('sentiment_model.pkl', 'wb') as f: pickle.dump(sent_model, f)
with open('tfidf_sent.pkl', 'wb') as f:      pickle.dump(tfidf_sent, f)
print("💾 Saved: absa_predictions.csv + all ABSA model pickle files")
print("✅ Section 3 — ABSA complete!")

---
## 📩 Section 4 — Personalized Message Generator

### What & Why
This section ties everything together. For each at-risk customer, we combine:
- Their **segment** (Section 1) — who they are
- Their **churn probability** (Section 2) — how urgent the outreach is
- Their **sentiment** (Section 3) — what they liked/disliked
- Their **favorite menu items** — personalization anchor
- **Campbell's menu** — what to promote

We call the **Groq API (LLaMA 3.3 70B)** to generate three message formats:
- 📱 **SMS** — max 160 characters, punchy
- 📧 **Email** — warm, 3-4 paragraphs, personal
- 🔔 **App notification** — max 80 characters, exciting

### Discount Strategy
| Risk Level | Discount |
|---|---|
| High | 20% off |
| Medium | 15% off |
| Low | 10% off |


### 4.1 — Imports & Configuration

In [ ]:
import json
import requests
import re
from openpyxl import load_workbook

GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"   # ← get from console.groq.com
GROQ_MODEL   = "llama-3.3-70b-versatile"
MAX_TOKENS   = 1000

DISCOUNT_MAP = {'High': 20, 'Medium': 15, 'Low': 10}
print("✅ Configuration set")

### 4.2 — Load Menu Data

In [ ]:
def load_menu(menu_path='Campbell_Menu_Data_-_2.xlsx'):
    wb      = load_workbook(menu_path, read_only=True)
    ws      = wb.active
    rows    = list(ws.iter_rows(values_only=True))
    menu_df = pd.DataFrame(rows[1:], columns=rows[0])
    food_cats = ['Signature Flights','Brunch Food','Entrées','Desserts',
                 'Salads','Burgers & Sandwiches','Kids Menu',
                 'Sides Dinner','Sides Brunch','Weekly Specials']
    food_menu = menu_df[menu_df['Category'].isin(food_cats)].dropna(subset=['itemName','itemPrice'])
    print(f"✅ Menu items loaded: {len(food_menu)} food items")
    return food_menu

menu_df = load_menu()
menu_df.head()

### 4.3 — Groq API Call

In [ ]:
def call_groq(prompt):
    """Call Groq API with JSON mode enforced."""
    headers = {
        "Content-Type" : "application/json",
        "Authorization": f"Bearer {GROQ_API_KEY}"
    }
    body = {
        "model"          : GROQ_MODEL,
        "max_tokens"     : MAX_TOKENS,
        "temperature"    : 0.7,
        "response_format": {"type": "json_object"},  # force pure JSON output
        "messages"       : [
            {
                "role"   : "system",
                "content": (
                    "You are a marketing assistant for Campbell's Restaurant. "
                    "You MUST respond with valid JSON only. "
                    "No markdown, no code fences, no extra text — pure JSON."
                )
            },
            {"role": "user", "content": prompt}
        ]
    }
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers=headers, json=body, timeout=30
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content']

print("✅ Groq API caller defined")

### 4.4 — Prompt Builder

In [ ]:
def build_prompt(customer, menu_df, aspects=None, sentiments=None):
    """
    Build a personalized prompt for a customer using their full profile.
    Combines segment, churn risk, sentiment, favorites, and menu highlights.
    """
    discount = DISCOUNT_MAP.get(str(customer['Risk_Level']), 15)
    liked    = [a for a,s in zip(aspects or [], sentiments or []) if s == 'positive']
    disliked = [a for a,s in zip(aspects or [], sentiments or []) if s == 'negative']
    highlights = menu_df.sample(min(3, len(menu_df)))[['itemName','itemPrice','Category']].to_dict('records')

    return f"""You are a warm, friendly marketing assistant for Campbell's Restaurant.

CUSTOMER PROFILE:
- Segment            : {customer['Segment']}
- Days since last visit: {int(customer['Recency'])} days
- Total visits       : {int(customer['Frequency'])}
- Total spent        : ${float(customer['Monetary']):.2f}
- Churn risk         : {customer['Risk_Level']}
- Liked aspects      : {liked if liked else 'unknown'}
- Disliked aspects   : {disliked if disliked else 'none noted'}
- Favorite items     : {customer.get('Favorite_Items', [])}

MENU HIGHLIGHTS:
{json.dumps(highlights, indent=2)}

DISCOUNT TO OFFER: {discount}% off next visit

Return ONLY this JSON:
{{
  "sms": "...",
  "email": {{"subject": "...", "body": "..."}},
  "app_notification": "..."
}}

RULES:
- SMS: max 160 chars, casual and punchy
- Email: warm, personal, 3-4 short paragraphs
- App notification: max 80 chars, exciting
- Tone: Lost→urgent, New→welcoming, Occasional→appreciative, Regular→VIP
- NEVER mention churn, AI, or risk scores"""

print("✅ Prompt builder defined")

### 4.5 — Generate Messages for At-Risk Customers

In [ ]:
def generate_messages(profiles, menu_df, top_n=10):
    at_risk = (
        profiles[profiles['Risk_Level'] == 'High']
        .sort_values('Churn_Probability', ascending=False)
        .head(top_n)
    )
    print(f"🎯 Generating messages for top {len(at_risk)} at-risk customers...\n")
    results = []

    for _, customer in at_risk.iterrows():
        cid = customer['Last 4 Card Digits']
        print(f"  Customer {cid} | {customer['Segment']} | Risk: {customer['Risk_Level']} | Prob: {customer['Churn_Probability']:.2f}")

        prompt   = build_prompt(customer, menu_df)
        response = call_groq(prompt)

        try:
            clean      = re.sub(r'```json|```', '', response).strip()
            json_match = re.search(r'\{.*\}', clean, re.DOTALL)
            if json_match: clean = json_match.group(0)
            msgs = json.loads(clean)
        except (json.JSONDecodeError, AttributeError):
            msgs = {"sms": response[:160], "email": {}, "app_notification": response[:80]}

        results.append({
            'Customer_ID'      : cid,
            'Segment'          : customer['Segment'],
            'Risk_Level'       : customer['Risk_Level'],
            'Churn_Probability': round(float(customer['Churn_Probability']), 4),
            'Recency_Days'     : int(customer['Recency']),
            'Discount_Offered' : f"{DISCOUNT_MAP.get(str(customer['Risk_Level']), 15)}%",
            'SMS'              : msgs.get('sms', ''),
            'Email_Subject'    : msgs.get('email', {}).get('subject', ''),
            'Email_Body'       : msgs.get('email', {}).get('body', ''),
            'App_Notification' : msgs.get('app_notification', '')
        })
        print(f"    ✅ SMS: {msgs.get('sms','')[:80]}...")

    return pd.DataFrame(results)

# ── Run ──
# Make sure final_df is available from Section 2
messages_df = generate_messages(final_df, menu_df, top_n=10)
messages_df

### 4.6 — Save Output

In [ ]:
messages_df.to_csv('personalized_messages.csv', index=False)
print("💾 Saved: personalized_messages.csv")
print("\n✅ Section 4 — Message Generation complete!")

---
## ✅ Pipeline Summary

| Section | Model | Output Files |
|---|---|---|
| 1. Customer Segmentation | KMeans (k=4) | `rfm_segments.csv`, `kmeans_model.pkl`, `scaler.pkl`, `cluster_map.pkl` |
| 2. Churn Prediction | Two-Tier XGBoost | `churn_scores_final.csv`, `churn_model_tier2.pkl`, `churn_features.pkl` |
| 3. ABSA | TF-IDF + Logistic Regression | `absa_predictions.csv`, `aspect_model.pkl`, `sentiment_model.pkl`, `tfidf_aspect.pkl`, `tfidf_sent.pkl`, `mlb.pkl` |
| 4. Message Generator | Groq LLaMA 3.3 70B | `personalized_messages.csv` |

### Key Design Decisions
- **Time-based split** in churn prediction prevents data leakage
- **Two-tier churn model** is honest — rule-based where data is insufficient, ML where there's enough signal
- **Segment as feature** in XGBoost connects the two models, boosting AUC from 0.70 → 0.84
- **JSON mode** in Groq API ensures consistent, parseable output
- **All models saved as pickle files** — ready to load in FastAPI endpoints

### Next Steps
- Wrap models in **FastAPI** backend
- Build **Lovable** frontend dashboard
- Deploy on **Vercel** (frontend) + **Railway** (backend)
- Connect **Supabase** for customer data persistence
